# Classic ML Baseline (Language-Agnostic)

This notebook trains a **classical ML** classifier for Human vs AI code detection using only `code` and `label`.

Design goal: generalize better to unseen languages by relying on character/subword patterns in code rather than language metadata.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import re
from pathlib import Path

import numpy as np
import pandas as pd
from scipy import sparse

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.calibration import CalibratedClassifierCV
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.svm import LinearSVC
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    roc_auc_score,
    f1_score,
    precision_score,
    recall_score,
    balanced_accuracy_score,
)

In [ ]:
# Update this path if needed
dataset_folder = './../datasets/Task_A'
working_dir = Path('./outputs')
working_dir.mkdir(parents=True, exist_ok=True)
submission_path = working_dir / 'classic_ml_submission.csv'

In [ ]:
train_df = pd.read_parquet(f'{dataset_folder}/train.parquet')
val_df = pd.read_parquet(f'{dataset_folder}/validation.parquet')
test_df = pd.read_parquet(f'{dataset_folder}/test.parquet')

print('Train shape:', train_df.shape)
print('Validation shape:', val_df.shape)
print('Test shape:', test_df.shape)

In [ ]:
# Keep only code + label for training/evaluation
train_ml = train_df[['code', 'label']].copy()
val_ml = val_df[['code', 'label']].copy()

train_ml['code'] = train_ml['code'].fillna('').astype(str)
val_ml['code'] = val_ml['code'].fillna('').astype(str)

train_ml = train_ml[train_ml['label'].notna()].copy()
val_ml = val_ml[val_ml['label'].notna()].copy()

train_ml['label'] = train_ml['label'].astype(int)
val_ml['label'] = val_ml['label'].astype(int)

X_train = train_ml['code']
y_train = train_ml['label']
X_val = val_ml['code']
y_val = val_ml['label']

print('Train samples:', len(X_train))
print('Validation samples:', len(X_val))

In [ ]:
# Stronger language-agnostic feature extractor: char n-grams + token n-grams + style features
TOKEN_PATTERN = re.compile(r'\w+|[^\w\s]', flags=re.UNICODE)
IDENTIFIER_PATTERN = re.compile(r'[A-Za-z_][A-Za-z0-9_]*')
COMMENT_PATTERN = re.compile(r'(^\s*#.*$|//.*$|/\*.*?\*/)', flags=re.MULTILINE | re.DOTALL)


class StyleFeatureExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        rows = []
        for text in X:
            code = '' if pd.isna(text) else str(text)
            lines = code.splitlines()
            non_empty_lines = [line for line in lines if line.strip()]

            tokens = TOKEN_PATTERN.findall(code)
            token_count = len(tokens)
            char_count = len(code)

            identifiers = IDENTIFIER_PATTERN.findall(code)
            avg_identifier_len = (
                float(np.mean([len(identifier) for identifier in identifiers]))
                if identifiers
                else 0.0
            )

            comments = COMMENT_PATTERN.findall(code)
            comment_chars = sum(len(comment) for comment in comments)
            comment_ratio = comment_chars / max(char_count, 1)

            line_lengths = [len(line) for line in non_empty_lines]
            avg_line_len = float(np.mean(line_lengths)) if line_lengths else 0.0
            std_line_len = float(np.std(line_lengths)) if line_lengths else 0.0

            blank_line_ratio = (
                (len(lines) - len(non_empty_lines)) / max(len(lines), 1)
                if lines
                else 0.0
            )

            punct_count = sum(1 for token in tokens if re.fullmatch(r'[^\w\s]', token))
            punct_ratio = punct_count / max(token_count, 1)

            uppercase_ratio = sum(1 for ch in code if ch.isupper()) / max(char_count, 1)
            digit_ratio = sum(1 for ch in code if ch.isdigit()) / max(char_count, 1)

            rows.append([
                token_count,
                char_count,
                avg_identifier_len,
                comment_ratio,
                avg_line_len,
                std_line_len,
                blank_line_ratio,
                punct_ratio,
                uppercase_ratio,
                digit_ratio,
            ])

        return sparse.csr_matrix(np.asarray(rows, dtype=np.float32))


char_vectorizer = TfidfVectorizer(
    analyzer='char_wb',
    ngram_range=(3, 6),
    min_df=2,
    max_features=320_000,
    sublinear_tf=True,
    lowercase=False,
    strip_accents=None
)

word_vectorizer = TfidfVectorizer(
    analyzer='word',
    token_pattern=r'\b\w+\b|[^\w\s]',
    ngram_range=(1, 3),
    min_df=2,
    max_features=180_000,
    sublinear_tf=True,
    lowercase=False
)

style_features = StyleFeatureExtractor()

features = FeatureUnion([
    ('char_tfidf', char_vectorizer),
    ('word_tfidf', word_vectorizer),
    ('style', style_features),
])

base_svm = LinearSVC(
    C=2.0,
    class_weight='balanced',
    random_state=42
)

clf = CalibratedClassifierCV(
    estimator=base_svm,
    method='sigmoid',
    cv=3
)

pipeline = Pipeline([
    ('features', features),
    ('clf', clf),
])

In [ ]:
pipeline.fit(X_train, y_train)

val_prob = pipeline.predict_proba(X_val)[:, 1]

thresholds = np.linspace(0.20, 0.80, 121)
best_threshold = 0.5
best_f1 = -1.0

for threshold in thresholds:
    pred = (val_prob >= threshold).astype(int)
    f1 = f1_score(y_val, pred, zero_division=0)
    if f1 > best_f1:
        best_f1 = f1
        best_threshold = float(threshold)

val_pred = (val_prob >= best_threshold).astype(int)

acc = accuracy_score(y_val, val_pred)
precision = precision_score(y_val, val_pred, zero_division=0)
recall = recall_score(y_val, val_pred, zero_division=0)
f1 = f1_score(y_val, val_pred, zero_division=0)
auc = roc_auc_score(y_val, val_prob)
bal_acc = balanced_accuracy_score(y_val, val_pred)

print(f'Best threshold on validation: {best_threshold:.3f}')
print(f'Validation Accuracy: {acc:.4f}')
print(f'Validation Balanced Accuracy: {bal_acc:.4f}')
print(f'Validation Precision: {precision:.4f}')
print(f'Validation Recall: {recall:.4f}')
print(f'Validation F1: {f1:.4f}')
print(f'Validation ROC-AUC: {auc:.4f}')

print('\nClassification report:')
print(classification_report(y_val, val_pred, digits=4, zero_division=0))

print('Confusion matrix [ [TN, FP], [FN, TP] ]:')
print(confusion_matrix(y_val, val_pred))

FINAL_THRESHOLD = best_threshold

In [ ]:
full_train = pd.concat([train_ml, val_ml], ignore_index=True)
pipeline.fit(full_train['code'], full_train['label'])

if 'FINAL_THRESHOLD' not in globals():
    FINAL_THRESHOLD = 0.5
    print('FINAL_THRESHOLD was not found; fallback to 0.5')

test_code = test_df['code'].fillna('').astype(str)
test_prob = pipeline.predict_proba(test_code)[:, 1]
test_pred = (test_prob >= FINAL_THRESHOLD).astype(int)

submission = pd.DataFrame({
    'ID': test_df['ID'].values,
    'label': test_pred,
})

submission.to_csv(submission_path, index=False)
print(f'Saved submission: {submission_path}')

display(submission.head())